In [12]:
"""
Animal Facts Generator — Gradio UI (v2, improved design)
----------------------------------------------------------
Wraps a LangChain + Groq (ChatGroq) chain in a polished, horizontal Gradio interface.

Setup:
    pip install gradio langchain langchain-groq langchain-core

    Set your Groq API key as an environment variable before running:
        export GROQ_API_KEY="your_key_here"      (macOS/Linux)
        setx GROQ_API_KEY "your_key_here"         (Windows)

Run:
    python animal_facts_app_v2.py
"""

import os
import gradio as gr
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ---------------------------------------------------------------------------
# LangChain setup
# ---------------------------------------------------------------------------

MODEL_NAME = "openai/gpt-oss-120b"


def build_chain():
    model = ChatGroq(
        model=MODEL_NAME,
        groq_api_key=os.getenv("GROQ_API_KEY"),
    )

    prompt_template = ChatPromptTemplate.from_messages(
        [
            ("system", "You are a facts expert who knows facts about {animal}."),
            ("human", "Tell me {fact_count} facts."),
        ]
    )

    return prompt_template | model | StrOutputParser()


_chain = None


def get_chain():
    """Lazily build the chain so a missing API key doesn't crash app startup."""
    global _chain
    if _chain is None:
        _chain = build_chain()
    return _chain


# ---------------------------------------------------------------------------
# Core inference function
# ---------------------------------------------------------------------------

def generate_facts(animal: str, fact_count: int):
    if not animal or not animal.strip():
        return "⚠️ Please enter an animal name."

    if not os.getenv("GROQ_API_KEY"):
        return (
            "⚠️ No GROQ_API_KEY found in your environment.\n\n"
            "Set it with:\n"
            "  export GROQ_API_KEY=\"your_key_here\"   (macOS/Linux)\n"
            "  setx GROQ_API_KEY \"your_key_here\"      (Windows)\n"
            "then restart the app."
        )

    try:
        chain = get_chain()
        result = chain.invoke({
            "animal": animal.strip(),
            "fact_count": int(fact_count),
        })
        return result
    except Exception as e:
        return f"❌ An error occurred while generating facts:\n\n{e}"


def clear_fields():
    return "", 3, ""


# ---------------------------------------------------------------------------
# Gradio UI
# ---------------------------------------------------------------------------

CUSTOM_CSS = """
.gradio-container {max-width: 1100px !important; margin: auto;}
footer {visibility: hidden}

#header-banner {
    background: linear-gradient(135deg, #4f46e5 0%, #7c3aed 50%, #a855f7 100%);
    border-radius: 16px;
    padding: 24px 26px;
    margin-bottom: 18px;
    box-shadow: 0 8px 24px rgba(79, 70, 229, 0.25);
}
#header-banner h1 {
    color: #ffffff !important;
    font-size: 1.6rem;
    margin: 0 0 6px 0;
    text-align: center;
}
#header-banner p {
    color: #e0e7ff !important;
    text-align: center;
    margin: 0;
    font-size: 0.92rem;
}

#input-card, #output-card {
    background: var(--block-background-fill);
    border: 1px solid var(--border-color-primary);
    border-radius: 14px;
    padding: 18px 20px;
    box-shadow: 0 2px 10px rgba(0,0,0,0.05);
}

#generate-btn {
    background: linear-gradient(135deg, #4f46e5, #7c3aed) !important;
    color: white !important;
    border: none !important;
    font-weight: 600 !important;
    font-size: 1.05rem !important;
    border-radius: 10px !important;
    box-shadow: 0 4px 14px rgba(124, 58, 237, 0.35) !important;
    transition: transform 0.15s ease, box-shadow 0.15s ease !important;
}
#generate-btn:hover {
    transform: translateY(-2px);
    box-shadow: 0 6px 18px rgba(124, 58, 237, 0.45) !important;
}

#clear-btn {
    border-radius: 10px !important;
    font-weight: 500 !important;
    border: 1px solid var(--border-color-primary) !important;
}

#output-box textarea {
    font-size: 1.02rem !important;
    line-height: 1.6 !important;
}

#footer-note {
    text-align: center;
    color: #9ca3af;
    font-size: 0.85rem;
    margin-top: 18px;
}
"""

THEME = gr.themes.Soft(
    primary_hue="indigo",
    secondary_hue="slate",
    font=[gr.themes.GoogleFont("Inter"), "ui-sans-serif", "system-ui"],
)

with gr.Blocks(title="Animal Facts Generator") as demo:

    with gr.Row(equal_height=True):
        with gr.Column(scale=1, min_width=380, elem_id="input-card"):
            gr.HTML(
                """
                <div id="header-banner">
                    <h1>🐾 Animal Facts Generator</h1>
                    <p>Ask an AI facts expert about any animal — powered by LangChain + Groq</p>
                </div>
                """
            )

            gr.Markdown("### 🔍 Ask a Question")

            animal_input = gr.Textbox(
                label="Animal",
                placeholder="e.g. elephant, octopus, red panda…",
                value="elephant",
            )

            fact_slider = gr.Slider(
                minimum=1,
                maximum=10,
                value=3,
                step=1,
                label="Number of facts",
            )

            with gr.Row():
                clear_btn = gr.Button("🗑️ Clear", elem_id="clear-btn", scale=1)
                generate_btn = gr.Button(
                    "✨ Generate Facts", elem_id="generate-btn", variant="primary", scale=2
                )

            gr.Examples(
                examples=[
                    ["elephant", 3],
                    ["octopus", 5],
                    ["platypus", 4],
                    ["snow leopard", 2],
                ],
                inputs=[animal_input, fact_slider],
                label="Try an example",
            )

        with gr.Column(scale=1, min_width=380, elem_id="output-card"):
            gr.Markdown("### 📋 Results")
            output_box = gr.Textbox(
                label="",
                lines=22,
                interactive=False,
                placeholder="Your facts will appear here...",
                elem_id="output-box",
                show_label=False,
            )

    gr.HTML(f"<div id='footer-note'>Powered by LangChain + Groq ({MODEL_NAME})</div>")

    generate_btn.click(
        fn=generate_facts,
        inputs=[animal_input, fact_slider],
        outputs=output_box,
    )

    animal_input.submit(
        fn=generate_facts,
        inputs=[animal_input, fact_slider],
        outputs=output_box,
    )

    clear_btn.click(
        fn=clear_fields,
        inputs=[],
        outputs=[animal_input, fact_slider, output_box],
    )


if __name__ == "__main__":
    demo.launch(theme=THEME, css=CUSTOM_CSS)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e38d1fea7909f21c2d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
